# Extreme periods

Clustering finds *typical* behaviour, so a once-a-year peak day tends to be blended
into an average and lost. For capacity sizing or reliability that single day is
often the one that matters. `ExtremeConfig` forces chosen extremes to be kept
exactly, alongside the normal clustering.

In [ ]:
import pandas as pd
import plotly.io as pio

import tsam

pio.renderers.default = "notebook_connected"

raw = pd.read_csv("testdata.csv", index_col=0, parse_dates=True)
data = raw.loc["2010-01-01":"2010-02-11"]  # six weeks of hourly data

## The peak gets averaged away

A plain aggregation clips the annual peak — the highest typical-day value sits well
below the real maximum:

In [ ]:
base = tsam.aggregate(data, n_clusters=6, period_duration="1D")
print(f"original peak Load:    {data['Load'].max():.1f}")
print(f"typical-period peak:   {base.cluster_representatives['Load'].max():.1f}")

## Keep it exactly

`ExtremeConfig` selects extreme **periods** to preserve. You can target the period
containing the single highest/lowest value (`max_value`/`min_value`) or the
highest/lowest period total (`max_period`/`min_period`):

In [ ]:
from tsam import ExtremeConfig

kept = tsam.aggregate(
    data,
    n_clusters=6,
    period_duration="1D",
    extremes=ExtremeConfig(method="new_cluster", max_value=["Load"]),
)
print(f"clusters: {base.n_clusters} -> {kept.n_clusters}")
print(f"typical-period peak now: {kept.cluster_representatives['Load'].max():.1f}")

## How the extreme enters: `method`

- **`new_cluster`** — add the extreme period as its own cluster (kept exactly,
  never blended). Recommended.
- **`append`** — add it as an extra typical period.
- **`replace`** — substitute the most similar existing cluster (cluster count
  unchanged).

In [ ]:
for m in ["new_cluster", "append", "replace"]:
    r = tsam.aggregate(
        data,
        n_clusters=6,
        period_duration="1D",
        extremes=ExtremeConfig(method=m, max_value=["Load"]),
    )
    print(
        f"{m:12s} -> {r.n_clusters} clusters, peak {r.cluster_representatives['Load'].max():.1f}"
    )

## Several extremes at once

Request as many as you need — e.g. the peak-demand day *and* the coldest day. A
period that is extreme for more than one criterion is added only once:

In [ ]:
multi = tsam.aggregate(
    data,
    n_clusters=6,
    period_duration="1D",
    extremes=ExtremeConfig(method="new_cluster", max_value=["Load"], min_value=["T"]),
)
print(f"clusters: {base.n_clusters} -> {multi.n_clusters}")

Extreme days stand in for just themselves, so they carry a small occurrence count
next to the typical clusters:

In [ ]:
kept.plot.cluster_counts()

A softer alternative that biases *every* representative toward extremes (rather
than adding specific days) is the `maxoid` / `minmax_mean` representation — see
[Representations](representations.ipynb).